# M5 · SQL fundamentos

**Bootcamp de Datos para Funcionarios Públicos — Formación Pública**
Módulo compartido · Tronco común (Semana 6) · Se ejecuta en Google Colab.

## ¿Qué vas a lograr?
Hasta ahora cargabas **toda** una tabla en Python y la filtrabas con pandas. Eso funciona con tablas chicas, pero las bases de datos del Estado tienen **millones de filas**: traerlas enteras es lento e inviable. **SQL** (*Structured Query Language*) es el lenguaje con el que le pides a una base de datos **justo lo que necesitas**, y ella te devuelve solo eso. Es la herramienta más usada para consultar datos en el mundo, y la que comparten las dos líneas del bootcamp.

**Competencia de salida:** escribir consultas SQL para seleccionar columnas, filtrar filas, ordenar resultados y resumir datos por grupos.

**Entregable:** que las cinco celdas de chequeo muestren ✅.


## Los datos que vamos a usar
Cambiamos de tema (¡otra vez!): ahora trabajamos con **medio ambiente**. Usaremos el listado real de los **Parques Nacionales de Chile**, con su región, su año de declaración y su superficie en hectáreas.

> **Fuente:** Corporación Nacional Forestal (**CONAF**) — Sistema Nacional de Áreas Silvestres Protegidas del Estado (SNASPE). CONAF administra 110 áreas protegidas (46 parques nacionales, 45 reservas y 19 monumentos naturales), unos **18,8 millones de hectáreas**, el 21% del territorio. Aquí usamos 24 parques nacionales con datos reales de superficie y año.

La tabla se llama `parques` y tiene cuatro columnas: `nombre`, `region`, `anio` (año de declaración) y `superficie_ha` (superficie en hectáreas).


## Preparar la base de datos
SQL necesita una **base de datos** donde correr. Usaremos **SQLite**, un motor de base de datos que viene **incluido en Python** (módulo `sqlite3`): no hay que instalar nada y funciona perfecto en Colab.

Ejecuta la celda de abajo para preparar el entorno. Carga el archivo `parques.csv` y crea la tabla `parques` en una base de datos en memoria. También define una función `consultar(sql)` que ejecuta tu consulta y te muestra el resultado como una tabla de pandas. La usarás en todo el módulo.

*(Nota: Para este módulo, el archivo `parques.csv` ya viene guardado de forma fija y estática en la carpeta de tu proyecto. Fuente de los datos: Corporación Nacional Forestal (CONAF) — Sistema Nacional de Áreas Silvestres Protegidas del Estado (SNASPE))*

In [ ]:
import os
import urllib.request
import pandas as pd
import sqlite3

# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("parques.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/A4-sql-fundamentos/parques.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "parques.csv")
        print("parques.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar parques.csv automáticamente. Si estás en Colab, asegúrate de subirlo manualmente.")

df = pd.read_csv("parques.csv")

# Creamos una base de datos SQLite en memoria y cargamos la tabla "parques"
conn = sqlite3.connect(":memory:")
df.to_sql("parques", conn, index=False, if_exists="replace")

def consultar(sql):
    """Ejecuta una consulta SQL sobre la base y devuelve el resultado como tabla."""
    return pd.read_sql_query(sql, conn)

print("Base lista. La tabla 'parques' tiene", len(df), "filas.")
consultar("SELECT * FROM parques LIMIT 5")

## 1. ¿Qué es SQL y por qué usarlo?
Imagina una **biblioteca enorme**. Tienes dos formas de trabajar:
- **Como en pandas hasta ahora:** pedir que te traigan *todos* los libros a tu mesa y revisarlos uno por uno. Si hay millones, no cabe en la mesa.
- **Como con SQL:** entregar una **boleta de pedido** ("quiero los libros de historia, publicados después de 2010, ordenados por autor") y que el bibliotecario te traiga **solo esos**.

SQL es esa boleta de pedido. Una consulta (*query*) describe **qué** quieres, no **cómo** buscarlo: la base de datos se encarga de lo demás. Por eso se dice que SQL es un lenguaje **declarativo**.

Toda consulta básica tiene esta forma:

```sql
SELECT columnas
FROM   tabla
WHERE  condición      -- (opcional) filtra filas
ORDER BY columna      -- (opcional) ordena el resultado
```

Las palabras `SELECT`, `FROM`, etc. son **palabras clave** de SQL. Por costumbre se escriben en MAYÚSCULAS para distinguirlas, aunque SQL no lo exige.


## 2. SELECT y FROM: elegir columnas
`SELECT` indica **qué columnas** quieres; `FROM`, de **qué tabla**.

- `SELECT * FROM parques` → trae **todas** las columnas (`*` significa "todo").
- `SELECT nombre, region FROM parques` → trae solo esas dos columnas.

Es el equivalente a elegir columnas en pandas (`df[["nombre", "region"]]`), pero pidiéndoselo a la base.


In [ ]:
# Todas las columnas (limitamos a 5 filas para no llenar la pantalla)
consultar("SELECT * FROM parques LIMIT 5")

In [ ]:
# Solo dos columnas
consultar("SELECT nombre, superficie_ha FROM parques LIMIT 5")

## 3. WHERE: filtrar filas
`WHERE` se queda solo con las filas que cumplen una **condición**. Los operadores más usados:

- `=` igual a (¡ojo! en SQL la igualdad es **un solo** `=`, no `==` como en Python),
- `>`, `<`, `>=`, `<=` mayor / menor,
- `<>` o `!=` distinto,
- `AND`, `OR` para combinar condiciones,
- `LIKE` para buscar texto parcial (`'%Torres%'` = "que contenga Torres").

⚠️ **Los textos van entre comillas simples**: `WHERE region = 'Magallanes'`. Si escribes `WHERE region = Magallanes` (sin comillas), SQL cree que `Magallanes` es otra columna y falla.


In [ ]:
# Parques con más de 200.000 hectáreas
consultar("SELECT nombre, superficie_ha FROM parques WHERE superficie_ha > 200000")

In [ ]:
# Parques de la región de La Araucanía declarados después de 1950
consultar("SELECT nombre, anio FROM parques WHERE region = 'La Araucanía' AND anio > 1950")

## 4. ORDER BY: ordenar el resultado
`ORDER BY` ordena las filas por una columna:

- `ORDER BY superficie_ha` → de menor a mayor (ascendente, es lo que asume por defecto).
- `ORDER BY superficie_ha DESC` → de mayor a menor (`DESC` = *descending*).

Y `LIMIT n` se queda solo con las primeras `n` filas. Combinados sirven para **rankings**: "los 3 parques más grandes".


In [ ]:
# Los 5 parques más grandes
consultar("SELECT nombre, superficie_ha FROM parques ORDER BY superficie_ha DESC LIMIT 5")

## 5. Funciones de agregación: resumir
Una **función de agregación** toma muchas filas y devuelve **un solo número**. Las cinco básicas:

- `COUNT(*)` → cuántas filas hay,
- `SUM(columna)` → suma,
- `AVG(columna)` → promedio,
- `MAX(columna)` / `MIN(columna)` → máximo / mínimo.

Puedes renombrar el resultado con `AS`: `SELECT COUNT(*) AS total`.


In [ ]:
# ¿Cuántos parques hay y cuánta superficie protegen en total?
consultar("SELECT COUNT(*) AS total_parques, SUM(superficie_ha) AS hectareas FROM parques")

## 6. GROUP BY: resumir **por grupo**
Lo más potente: `GROUP BY` aplica una función de agregación **a cada grupo** por separado. Por ejemplo, en vez de contar todos los parques juntos, los cuenta **por región**.

```sql
SELECT region, COUNT(*) AS n
FROM   parques
GROUP BY region
```

Esto devuelve una fila por región, con cuántos parques tiene cada una. Es el equivalente a `value_counts()` de pandas, pero mucho más flexible (puedes sumar, promediar, etc., por grupo).

> **Regla de oro de `GROUP BY`:** en el `SELECT` solo puedes poner las columnas por las que agrupas (`region`) y funciones de agregación (`COUNT`, `SUM`...). Poner una columna suelta que no está en el `GROUP BY` da resultados confusos o un error.


In [ ]:
# Cuántos parques y cuánta superficie por región, de mayor a menor
consultar("""
    SELECT region, COUNT(*) AS n_parques, SUM(superficie_ha) AS hectareas
    FROM parques
    GROUP BY region
    ORDER BY hectareas DESC
""")

## ⚠️ Errores típicos de principiante
- **Texto sin comillas.** `WHERE region = Magallanes` falla; debe ser `WHERE region = 'Magallanes'` (comillas simples).
- **Usar `==` para comparar.** En SQL la igualdad es **un solo** `=`. El `==` de Python aquí da error.
- **Olvidar el `FROM`.** Toda consulta que lee datos necesita decir de qué tabla (`FROM parques`).
- **Confundir mayúsculas en los textos.** `'magallanes'` no es igual a `'Magallanes'`: SQL compara el texto tal cual. Escríbelo igual que en los datos, o usa `UPPER(region) = 'MAGALLANES'`.
- **Esperar orden sin pedirlo.** Sin `ORDER BY`, la base puede devolver las filas en cualquier orden. Si quieres un ranking, ordénalo explícitamente.
- **Romper la regla de `GROUP BY`.** Seleccionar columnas sueltas que no están agrupadas ni agregadas.


---
# 🛠️ Ejercicios
Cinco ejercicios sobre la tabla real `parques`. En cada uno, **asigna tu consulta SQL a la variable indicada** (como texto, entre comillas). Luego ejecuta la celda de chequeo (✅ / ❌). La función `consultar(...)` ya está lista del setup.


## Ejercicio 01 — SELECT
Escribe en `sql_01` una consulta que devuelva **dos columnas**: `nombre` y `superficie_ha`, de **todos** los parques.

Pista: `SELECT columna1, columna2 FROM parques`.

In [ ]:
sql_01 = "SELECT nombre FROM parques"   # TODO: agrega la columna superficie_ha

consultar(sql_01)

In [ ]:
try:
    r = consultar(sql_01)
    assert list(r.columns) == ["nombre", "superficie_ha"], f"Esperaba columnas [nombre, superficie_ha], obtuve {list(r.columns)}"
    assert len(r) == 24, f"Esperaba 24 filas, obtuve {len(r)}"
    print("✅ Correcto. Ejercicio 01 superado: tu primer SELECT.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir sql_01:", e)
except Exception as e:
    print("❌ La consulta tiene un error de SQL. Revísala:", e)

## Ejercicio 02 — WHERE
Escribe en `sql_02` una consulta que devuelva el `nombre` de los parques de la **región de Aysén**.

Pista: usa `WHERE region = 'Aysén'` (¡comillas simples y acento!).

In [ ]:
sql_02 = "SELECT nombre FROM parques"   # TODO: filtra por la región de Aysén

consultar(sql_02)

In [ ]:
try:
    r = consultar(sql_02)
    assert len(r) == 2, f"Esperaba 2 parques en Aysén, obtuve {len(r)}"
    assert set(r["nombre"]) == {"Laguna San Rafael", "Patagonia"}, "No son los parques esperados de Aysén"
    print("✅ Correcto. Ejercicio 02 superado: filtraste con WHERE.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir sql_02:", e)
except KeyError:
    print("❌ Asegúrate de seleccionar la columna nombre.")
except Exception as e:
    print("❌ La consulta tiene un error de SQL. Revísala:", e)

## Ejercicio 03 — ORDER BY + LIMIT
Escribe en `sql_03` una consulta que devuelva los **3 parques más grandes** (columnas `nombre` y `superficie_ha`), del más grande al más chico.

Pista: `ORDER BY superficie_ha DESC` y `LIMIT 3`.

In [ ]:
sql_03 = "SELECT nombre, superficie_ha FROM parques LIMIT 3"   # TODO: ordena por superficie_ha DESC

consultar(sql_03)

In [ ]:
try:
    r = consultar(sql_03)
    assert len(r) == 3, f"Esperaba 3 filas, obtuve {len(r)}"
    assert r.iloc[0]["nombre"] == "Bernardo O'Higgins", "El parque más grande debería ir primero"
    assert r.iloc[1]["nombre"] == "Laguna San Rafael", "El segundo más grande no es el esperado"
    print("✅ Correcto. Ejercicio 03 superado: armaste un ranking.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir sql_03:", e)
except Exception as e:
    print("❌ La consulta tiene un error de SQL. Revísala:", e)

## Ejercicio 04 — GROUP BY
Escribe en `sql_04` una consulta que devuelva, **por región**, cuántos parques tiene cada una. Dos columnas: `region` y el conteo (llámalo `n`).

Pista: `SELECT region, COUNT(*) AS n FROM parques GROUP BY region`.

In [ ]:
sql_04 = "SELECT region, anio FROM parques GROUP BY region"   # TODO: cuenta los parques con COUNT(*) AS n

consultar(sql_04)

In [ ]:
try:
    r = consultar(sql_04)
    conteo = dict(zip(r.iloc[:, 0], r.iloc[:, 1]))
    assert len(r) == 12, f"Esperaba 12 regiones, obtuve {len(r)}"
    assert conteo["La Araucanía"] == 5, f"La Araucanía debería tener 5 parques, no {conteo.get('La Araucanía')}"
    assert conteo["Magallanes"] == 5, f"Magallanes debería tener 5 parques, no {conteo.get('Magallanes')}"
    print("✅ Correcto. Ejercicio 04 superado: ¡resumiste con GROUP BY!")
    mas = max(conteo, key=conteo.get)
    print(f"Regiones con más parques: La Araucanía y Magallanes, con {conteo[mas]} cada una.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir sql_04:", e)
except (KeyError, IndexError):
    print("❌ Devuelve dos columnas: la región y el conteo COUNT(*).")
except Exception as e:
    print("❌ La consulta tiene un error de SQL. Revísala:", e)

## Ejercicio 05 — Interpretación: ¿promedio o mediana?

Hasta aquí calculaste valores. Ahora toca **interpretarlos**: un número solo no dice nada si no sabes leerlo.

La superficie de los parques va desde unas pocas miles de hectáreas hasta varios **millones**. Vamos a comparar dos formas de resumir "el tamaño típico" de un parque:

- el **promedio** (`AVG`), que suma todo y reparte en partes iguales, y
- la **mediana**, el valor del medio cuando ordenas los parques de menor a mayor (la mitad queda abajo y la mitad arriba).

**Tu tarea:**
1. Calcula el **promedio** de `superficie_ha` y guárdalo en `promedio_ha`.
2. Calcula la **mediana** de `superficie_ha` y guárdala en `mediana_ha`.
3. Compara ambos números y **elige la interpretación correcta**, asignando la letra (como texto, ej. `"A"`) a la variable `conclusion`.

> Pista para la mediana: SQLite no tiene una función `MEDIAN` directa. Lo más simple es calcularla con pandas sobre el DataFrame `df` que ya está cargado: `df["superficie_ha"].median()`. El promedio puedes sacarlo con SQL (`SELECT AVG(superficie_ha) FROM parques`) o también con pandas (`df["superficie_ha"].mean()`).

**Opciones de interpretación:**

- **A.** El promedio y la mediana son casi iguales, así que la mayoría de los parques tiene un tamaño parecido al promedio.
- **B.** El promedio es **mucho mayor** que la mediana: unos pocos parques gigantes inflan el promedio, y la **mayoría** de los parques es **bastante más chica** que ese promedio. La mediana describe mejor el tamaño típico.
- **C.** La mediana es mucho mayor que el promedio, así que la mayoría de los parques es más grande de lo que sugiere el promedio.

> Reflexión libre (no se corrige): si tuvieras que reportarle a una autoridad "el tamaño típico de un parque nacional", ¿usarías el promedio o la mediana? ¿Por qué?

In [ ]:
# Calcula el promedio y la mediana de la superficie
promedio_ha = None   # TODO: promedio de superficie_ha (SQL con AVG, o df["superficie_ha"].mean())
mediana_ha = None    # TODO: mediana de superficie_ha (df["superficie_ha"].median())

# Compara los dos números y elige la interpretación correcta (A, B o C)
conclusion = None    # TODO: asigna la letra entre comillas, por ejemplo "A"

print("Promedio:", promedio_ha)
print("Mediana :", mediana_ha)
print("Conclusión:", conclusion)

In [ ]:
try:
    # Recomputamos los valores esperados directamente desde los datos
    prom_esperado = df["superficie_ha"].mean()
    med_esperado = df["superficie_ha"].median()

    assert promedio_ha is not None, "Aún no calculaste el promedio (promedio_ha sigue en None)."
    assert mediana_ha is not None, "Aún no calculaste la mediana (mediana_ha sigue en None)."
    assert abs(float(promedio_ha) - prom_esperado) < 1, (
        f"El promedio no coincide. Esperaba {prom_esperado:.1f}, obtuve {promedio_ha}."
    )
    assert abs(float(mediana_ha) - med_esperado) < 1, (
        f"La mediana no coincide. Esperaba {med_esperado:.1f}, obtuve {mediana_ha}."
    )

    # La conclusión correcta se deduce de comparar promedio vs mediana
    conclusion_correcta = "B" if prom_esperado > med_esperado * 1.5 else "A"
    assert conclusion is not None, "Aún no elegiste una interpretación (conclusion sigue en None)."
    assert str(conclusion).strip().upper() == conclusion_correcta, (
        "Esa no es la interpretación correcta. Fíjate: el promedio es "
        f"{prom_esperado/med_esperado:.1f} veces la mediana. ¿Qué dice eso de la forma de la distribución?"
    )

    print("✅ Correcto. Ejercicio 05 superado: interpretaste promedio vs mediana.")
    print(f"   Promedio = {prom_esperado:,.0f} ha · Mediana = {med_esperado:,.0f} ha")
    n_bajo = int((df["superficie_ha"] < prom_esperado).sum())
    print(f"   {n_bajo} de {len(df)} parques están POR DEBAJO del promedio: unos pocos gigantes lo inflan.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir alguna variable (promedio_ha, mediana_ha o conclusion):", e)
except Exception as e:
    print("❌ Algo falló. Revisa tu cálculo:", e)

---
## ¿Terminaste?
Si las cinco celdas de chequeo muestran ✅, completaste **M5**. 🎉

Aprendiste el corazón de SQL: **SELECT/FROM** (elegir columnas), **WHERE** (filtrar), **ORDER BY/LIMIT** (ordenar y rankear) y **COUNT/SUM/AVG + GROUP BY** (resumir por grupo). Con esto ya puedes consultar la mayoría de las bases de datos del Estado pidiendo justo lo que necesitas, sin traer tablas enteras.

SQL es además el cruce de caminos del bootcamp: las dos líneas lo profundizan distinto (**A8 · SQL analítico avanzado** y **D8 · SQL para features**). Pero antes, en **M6 · Estadística descriptiva**, volverás a pandas para aprender a describir un conjunto de datos con números: promedios, medianas, dispersión y cómo no dejarte engañar por un solo indicador.